In [46]:
import pandas as pd
from pathlib import Path
from countrycode import countrycode


In [47]:
base=pd.read_csv('data/data_clean/final.csv')
base["year"] = pd.to_datetime(base["year"].astype(str), format="%Y")
df= pd.read_stata('data/data_raw/GSDB_V4_dyadic.dta')

In [48]:
df

,case_id,sanctioning_state_iso3,sanctioning_state,sanctioned_state_iso3,sanctioned_state,year,arms,military,trade,descr_trade,financial,travel,other,target_mult,sender_mult,objective,success
0,471,AFG,Afghanistan,AGO,Angola,1993-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
1,471,AFG,Afghanistan,AGO,Angola,1994-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
2,471,AFG,Afghanistan,AGO,Angola,1995-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
3,471,AFG,Afghanistan,AGO,Angola,1996-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
4,"471,574",AFG,Afghanistan,AGO,Angola,1997-01-01,1,1,1,exp_part,0,1,1,0,1,"end_war,end_war","success_total,success_total"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159060,"1031,1047",ZWE,Zimbabwe,YEM,"Yemen, North",2019-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"
159061,"1047,1031",ZWE,Zimbabwe,YEM,"Yemen, North",2020-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"
159062,"1031,1047",ZWE,Zimbabwe,YEM,"Yemen, North",2021-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"
159063,"1031,1047",ZWE,Zimbabwe,YEM,"Yemen, North",2022-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"


In [24]:



# integer year (GSDB stores the year as a date like 1993-01-01)
yr = pd.to_datetime(df["year"]).dt.year

iso_cols  = ["sanctioning_state_iso3", "sanctioned_state_iso3"]
name_cols = ["sanctioning_state", "sanctioned_state"]

# snapshot the ORIGINAL codes — the drop/separate masks must test these,
# not values that a relabel below has already changed
s0 = df["sanctioning_state_iso3"].copy()
t0 = df["sanctioned_state_iso3"].copy()

# =====================================================================
# STEP 1 — mark rows to DROP  (row goes if EITHER side hits a drop rule)
# =====================================================================
def _drop(code):   # small mask helper, not the recoding logic itself
    return (
        (code == "")                                    # terrorist orgs / non-state
        | (code == "BYS")                               # Byelorussian SSR (UN-seat artifact)
        | ((code == "UKR") & (yr <= 1990))              # Ukrainian SSR seat
        | ((code == "SVU") & (yr >= 1990))              # USSR 1990-91 tail (Russia takes over)
        | ((code == "RUS") & (yr <  1990))              # pre-1990 RUS duplicates
        | ((code == "KSV") & (yr <= 2007))              # premature/erroneous Kosovo (case 663)
        | ((code == "ARM") & (yr <  1991))              # pre-independence 'bleed' below
        | ((code == "AZE") & (yr <  1991))
        | ((code == "EST") & (yr <  1991))
        | ((code == "LTU") & (yr <  1991))
        | ((code == "LVA") & (yr <  1991))
        | ((code == "BIH") & (yr <  1992))
        | ((code == "HRV") & (yr <  1992))
        | ((code == "SVN") & (yr <  1992))
        | ((code == "SGP") & (yr <  1965))
        | ((code == "GNB") & (yr <  1974))
        | ((code == "MOZ") & (yr <  1975))
    )
drop_mask = _drop(s0) | _drop(t0)

# =====================================================================
# STEP 2 — mark rows for a SEPARATE side table (kept, not merged)
# =====================================================================
separate = {"RHO", "PSE", "HKG", "ESH", "VAT", "MSR"}   # RHO kept aside (no Zimbabwe change)
separate_mask = (s0.isin(separate) | t0.isin(separate)) & ~drop_mask

# =====================================================================
# STEP 3 — relabel codes (and the two clean names), on both columns
# =====================================================================
# 3a. simple same-polity renames (all years)
df[iso_cols]  = df[iso_cols].replace({"DHY": "BEN", "HVO": "BFA"})
df[name_cols] = df[name_cols].replace({"Dahomey": "Benin", "Upper Volta": "Burkina Faso"})
# 3b. folds (all years)
df[iso_cols]  = df[iso_cols].replace({"GIB": "GBR", "COK": "NZL", "NIU": "NZL"})
# 3c. year-conditional relabels (test the original codes s0/t0, write to the live column)
for col, orig in [("sanctioning_state_iso3", s0), ("sanctioned_state_iso3", t0)]:
    df.loc[(orig == "SVU") & (yr <  1990), col] = "RUS"   # Soviet sanctions -> single RUS
    df.loc[(orig == "SRB") & (yr <= 2005), col] = "YUG"   # pre-2006 = Yugoslavia
    df.loc[(orig == "MNE") & (yr <= 2005), col] = "YUG"
    df.loc[(orig == "DDR") & (yr >= 1991), col] = "DEU"   # post-reunification residue
    df.loc[(orig == "CSK") & (yr >= 1993), col] = "CZE"   # post-dissolution residue
    df.loc[(orig == "CZE") & (yr <= 1992), col] = "CSK"   # successor code before the split
    df.loc[(orig == "SVK") & (yr <= 1992), col] = "CSK"
    df.loc[(orig == "VNM") & (yr >= 1976), col] = "VDR"   # unified Vietnam is VDR in base
    #df.loc[(orig == "ERI") & (yr <= 1992), col] = "ETH"   # Eritrea within Ethiopia pre-1993

# =====================================================================
# STEP 4 — split into buckets
# =====================================================================
panel = df[~drop_mask & ~separate_mask].copy()   # merge into final.csv (dedupe first!)
side  = df[separate_mask].copy()                  # RHO/PSE/HKG/ESH/VAT/MSR
# dropped = df[drop_mask]

print("merge:", len(panel), "| separate:", len(side), "| drop:", int(drop_mask.sum()))

merge: 146382 | separate: 2893 | drop: 9790


In [25]:
panel

,case_id,sanctioning_state_iso3,sanctioning_state,sanctioned_state_iso3,sanctioned_state,year,arms,military,trade,descr_trade,financial,travel,other,target_mult,sender_mult,objective,success
0,471,AFG,Afghanistan,AGO,Angola,1993-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
1,471,AFG,Afghanistan,AGO,Angola,1994-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
2,471,AFG,Afghanistan,AGO,Angola,1995-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
3,471,AFG,Afghanistan,AGO,Angola,1996-01-01,1,1,1,exp_part,0,0,0,0,1,end_war,success_total
4,"471,574",AFG,Afghanistan,AGO,Angola,1997-01-01,1,1,1,exp_part,0,1,1,0,1,"end_war,end_war","success_total,success_total"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159060,"1031,1047",ZWE,Zimbabwe,YEM,"Yemen, North",2019-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"
159061,"1047,1031",ZWE,Zimbabwe,YEM,"Yemen, North",2020-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"
159062,"1031,1047",ZWE,Zimbabwe,YEM,"Yemen, North",2021-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"
159063,"1031,1047",ZWE,Zimbabwe,YEM,"Yemen, North",2022-01-01,1,1,0,,1,1,0,0,1,"end_war,terrorism,human_rights,end_war,terrori...","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing"


In [26]:
base[base["iso3c"].isna()]['gw_name'].unique()

<StringArray>
['Abkhazia', 'South Ossetia', 'Zanzibar', 'Tibet']
Length: 4, dtype: str

In [53]:
received_sanctions = (
    panel.groupby(['sanctioned_state_iso3', 'year'])
    .agg(
        num_sanctioners=('sanctioning_state', 'nunique'),
        n_arms_received=('arms', 'sum'),
        n_military_received=('military', 'sum'),
        n_trade_received=('trade', 'sum'),
        n_financial_received=('financial', 'sum'),
        n_travel_received=('travel', 'sum'),
        n_other_received=('other', 'sum'),
    )
    .reset_index()
    .rename(columns={'sanctioned_state_iso3': 'country_iso3'})
)

outgoing_sanctions = (
    panel.groupby(['sanctioning_state_iso3', 'year'])
    .agg(
        num_targets_sanctioned=('sanctioned_state', 'nunique'),
        n_arms_sent=('arms', 'sum'),
        n_military_sent=('military', 'sum'),
        n_trade_sent=('trade', 'sum'),
        n_financial_sent=('financial', 'sum'),
        n_travel_sent=('travel', 'sum'),
        n_other_sent=('other', 'sum'),
    )
    .reset_index()
    .rename(columns={'sanctioning_state_iso3': 'country_iso3'})
)
svn_mask = (
    (received_sanctions["country_iso3"] == "SVN")
    & (received_sanctions["year"].dt.year.between(1991, 1996))
)

zero_cols = ["num_sanctioners", "n_arms_received", "n_military_received",
             "n_trade_received", "n_financial_received", "n_travel_received", "n_other_received"]

received_sanctions.loc[svn_mask, zero_cols] = 0

In [54]:
received_sanctions

,country_iso3,year,num_sanctioners,n_arms_received,n_military_received,n_trade_received,n_financial_received,n_travel_received,n_other_received
0,AFG,1979-01-01,1,0,1,0,1,0,0
1,AFG,1996-01-01,186,186,32,0,0,0,0
2,AFG,1997-01-01,186,186,32,0,0,0,0
3,AFG,1998-01-01,186,186,32,0,0,0,0
4,AFG,1999-01-01,189,189,32,1,188,0,188
...,...,...,...,...,...,...,...,...,...
5205,ZWE,2019-01-01,45,44,44,42,45,43,1
5206,ZWE,2020-01-01,45,44,44,42,45,43,1
5207,ZWE,2021-01-01,45,44,44,42,45,43,1
5208,ZWE,2022-01-01,45,44,44,42,45,43,1


In [55]:
outgoing_sanctions

,country_iso3,year,num_targets_sanctioned,n_arms_sent,n_military_sent,n_trade_sent,n_financial_sent,n_travel_sent,n_other_sent
0,AFG,1951-01-01,2,2,0,0,0,0,0
1,AFG,1952-01-01,2,2,0,0,0,0,0
2,AFG,1953-01-01,2,2,0,0,0,0,0
3,AFG,1954-01-01,1,1,0,0,0,0,0
4,AFG,1955-01-01,1,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
11329,ZWE,2019-01-01,15,13,9,2,11,12,3
11330,ZWE,2020-01-01,15,13,9,2,11,12,3
11331,ZWE,2021-01-01,17,13,9,2,12,12,4
11332,ZWE,2022-01-01,19,14,9,2,13,13,5


In [56]:
base["year"].dtype


dtype('<M8[us]')

In [57]:
received_sanctions["year"].dtype


dtype('<M8[s]')

In [58]:

outgoing_sanctions["year"].dtype

dtype('<M8[s]')

In [59]:
outgoing_sanctions

,country_iso3,year,num_targets_sanctioned,n_arms_sent,n_military_sent,n_trade_sent,n_financial_sent,n_travel_sent,n_other_sent
0,AFG,1951-01-01,2,2,0,0,0,0,0
1,AFG,1952-01-01,2,2,0,0,0,0,0
2,AFG,1953-01-01,2,2,0,0,0,0,0
3,AFG,1954-01-01,1,1,0,0,0,0,0
4,AFG,1955-01-01,1,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
11329,ZWE,2019-01-01,15,13,9,2,11,12,3
11330,ZWE,2020-01-01,15,13,9,2,11,12,3
11331,ZWE,2021-01-01,17,13,9,2,12,12,4
11332,ZWE,2022-01-01,19,14,9,2,13,13,5


In [60]:
merged = pd.merge(
    received_sanctions, base,
    how='left',
    left_on=['year', 'country_iso3'], right_on=['year', 'iso3c'],
    indicator=True
)
orphans = merged[merged['_merge'] == 'left_only']
orphans_check_recieved=orphans[['year', 'country_iso3','num_sanctioners']].drop_duplicates()
orphans_check_recieved

merged = pd.merge(
    outgoing_sanctions, base,
    how='left',
    left_on=['year', 'country_iso3'], right_on=['year', 'iso3c'],
    indicator=True
)
orphans = merged[merged['_merge'] == 'left_only']
orphans_check_outgoing=orphans[['year', 'country_iso3', 'num_targets_sanctioned']].drop_duplicates()
orphans_check_outgoing

,year,country_iso3,num_targets_sanctioned


In [61]:
orphans_check_recieved

,year,country_iso3,num_sanctioners
1570,1977-01-01,ERI,1
1571,1978-01-01,ERI,1
1572,1979-01-01,ERI,1
1573,1980-01-01,ERI,1
1574,1981-01-01,ERI,1
1575,1982-01-01,ERI,1
1576,1983-01-01,ERI,1
1577,1984-01-01,ERI,1
1578,1985-01-01,ERI,1
1579,1986-01-01,ERI,1


In [62]:
output_dir = Path('data/orphans')
output_dir.mkdir(parents=True, exist_ok=True)

orphans_check_outgoing.to_csv(output_dir / 'orphans_outgoing.csv', index=False)
orphans_check_recieved.to_csv(output_dir / 'orphans_recieved.csv', index=False)

In [63]:
orphans_check_recieved

,year,country_iso3,num_sanctioners
1570,1977-01-01,ERI,1
1571,1978-01-01,ERI,1
1572,1979-01-01,ERI,1
1573,1980-01-01,ERI,1
1574,1981-01-01,ERI,1
1575,1982-01-01,ERI,1
1576,1983-01-01,ERI,1
1577,1984-01-01,ERI,1
1578,1985-01-01,ERI,1
1579,1986-01-01,ERI,1


In [64]:
base

,gwcode,gw_name,microstate,year,ccode,gwc,iso3c,successor_iso3c,n_conflicts_all,n_conflicts_intra,...,euds,aeuds,polity2,v2x_polyarchy,mrgdppc,sd_mrgdppc,pwtrgdp,sd_pwtrgdp,pwtpop,sd_pwtpop
0,2,United States of America,0,1946-01-01,2.0,USA,USA,NaN,0,0,...,1.293985,0.486256,9.0,0.603,15916.605087,2494.189575,2.565007e+06,1.075414e+06,142.942214,9.874863
1,2,United States of America,0,1947-01-01,2.0,USA,USA,NaN,0,0,...,1.308359,0.500630,9.0,0.607,15691.605157,2443.671758,2.524909e+06,1.097010e+06,145.613447,10.302882
2,2,United States of America,0,1948-01-01,2.0,USA,USA,NaN,0,0,...,1.343539,0.535809,9.0,0.599,15634.165601,2397.095016,2.621770e+06,1.143350e+06,148.443589,10.571322
3,2,United States of America,0,1949-01-01,2.0,USA,USA,NaN,0,0,...,1.330836,0.523106,9.0,0.580,15774.791549,2415.503157,2.724009e+06,1.187962e+06,150.554205,10.712760
4,2,United States of America,0,1950-01-01,2.0,USA,USA,NaN,1,0,...,1.354015,0.546286,9.0,0.585,16610.071657,2605.983858,2.831568e+06,1.207133e+06,153.549515,10.469628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12305,990,Samoa/Western Samoa,1,2019-01-01,990.0,WSM,WSM,NaN,0,0,...,0.846640,0.038911,NaN,NaN,5280.997786,7602.198097,1.161916e+03,1.830815e+03,0.196232,0.015295
12306,990,Samoa/Western Samoa,1,2020-01-01,990.0,WSM,WSM,NaN,0,0,...,0.988814,0.181084,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12307,990,Samoa/Western Samoa,1,2021-01-01,990.0,WSM,WSM,NaN,0,0,...,1.327329,0.519600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12308,990,Samoa/Western Samoa,1,2022-01-01,990.0,WSM,WSM,NaN,0,0,...,1.008024,0.200294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [65]:
dup_mask = received_sanctions.duplicated(subset=["year", "country_iso3"], keep=False)
duplicates_sanct=received_sanctions[dup_mask].sort_values(["country_iso3", "year"])
duplicates_sanct

,country_iso3,year,num_sanctioners,n_arms_received,n_military_received,n_trade_received,n_financial_received,n_travel_received,n_other_received


In [66]:
duplicates_sanct

,country_iso3,year,num_sanctioners,n_arms_received,n_military_received,n_trade_received,n_financial_received,n_travel_received,n_other_received


In [67]:


base_recieved=pd.merge(base, received_sanctions, how='left',left_on=['year', 'iso3c'], right_on=['year', 'country_iso3'])
base_recieved

,gwcode,gw_name,microstate,year,ccode,gwc,iso3c,successor_iso3c,n_conflicts_all,n_conflicts_intra,...,pwtpop,sd_pwtpop,country_iso3,num_sanctioners,n_arms_received,n_military_received,n_trade_received,n_financial_received,n_travel_received,n_other_received
0,2,United States of America,0,1946-01-01,2.0,USA,USA,NaN,0,0,...,142.942214,9.874863,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,United States of America,0,1947-01-01,2.0,USA,USA,NaN,0,0,...,145.613447,10.302882,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,United States of America,0,1948-01-01,2.0,USA,USA,NaN,0,0,...,148.443589,10.571322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,United States of America,0,1949-01-01,2.0,USA,USA,NaN,0,0,...,150.554205,10.712760,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,United States of America,0,1950-01-01,2.0,USA,USA,NaN,1,0,...,153.549515,10.469628,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12305,990,Samoa/Western Samoa,1,2019-01-01,990.0,WSM,WSM,NaN,0,0,...,0.196232,0.015295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12306,990,Samoa/Western Samoa,1,2020-01-01,990.0,WSM,WSM,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12307,990,Samoa/Western Samoa,1,2021-01-01,990.0,WSM,WSM,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12308,990,Samoa/Western Samoa,1,2022-01-01,990.0,WSM,WSM,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [68]:
cols_to_fill = [c for c in list(outgoing_sanctions.columns) + list(received_sanctions.columns)
                 if c not in ['country', 'country_iso3', 'year']]
cols_to_fill

['num_targets_sanctioned',
 'n_arms_sent',
 'n_military_sent',
 'n_trade_sent',
 'n_financial_sent',
 'n_travel_sent',
 'n_other_sent',
 'num_sanctioners',
 'n_arms_received',
 'n_military_received',
 'n_trade_received',
 'n_financial_received',
 'n_travel_received',
 'n_other_received']

In [69]:
base_recieved_outgoing=pd.merge(base_recieved, outgoing_sanctions, how='left',left_on=['year', 'iso3c'], right_on=['year', 'country_iso3'])
base_recieved_outgoing=base_recieved_outgoing.drop(columns=['country_iso3_x', 'country_iso3_y'])
 # replace with your actual column names
base_recieved_outgoing[cols_to_fill] = base_recieved_outgoing[cols_to_fill].fillna(0)
base_recieved_outgoing

,gwcode,gw_name,microstate,year,ccode,gwc,iso3c,successor_iso3c,n_conflicts_all,n_conflicts_intra,...,n_financial_received,n_travel_received,n_other_received,num_targets_sanctioned,n_arms_sent,n_military_sent,n_trade_sent,n_financial_sent,n_travel_sent,n_other_sent
0,2,United States of America,0,1946-01-01,2.0,USA,USA,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,United States of America,0,1947-01-01,2.0,USA,USA,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,United States of America,0,1948-01-01,2.0,USA,USA,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2,United States of America,0,1949-01-01,2.0,USA,USA,NaN,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2,United States of America,0,1950-01-01,2.0,USA,USA,NaN,1,0,...,0.0,0.0,0.0,11.0,0.0,0.0,10.0,2.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12305,990,Samoa/Western Samoa,1,2019-01-01,990.0,WSM,WSM,NaN,0,0,...,0.0,0.0,0.0,15.0,13.0,9.0,2.0,11.0,12.0,3.0
12306,990,Samoa/Western Samoa,1,2020-01-01,990.0,WSM,WSM,NaN,0,0,...,0.0,0.0,0.0,15.0,13.0,9.0,2.0,11.0,12.0,3.0
12307,990,Samoa/Western Samoa,1,2021-01-01,990.0,WSM,WSM,NaN,0,0,...,0.0,0.0,0.0,16.0,13.0,9.0,2.0,12.0,12.0,3.0
12308,990,Samoa/Western Samoa,1,2022-01-01,990.0,WSM,WSM,NaN,0,0,...,0.0,0.0,0.0,17.0,14.0,9.0,2.0,13.0,13.0,3.0


In [70]:
output_dir = Path('data/data_clean')
output_dir.mkdir(parents=True, exist_ok=True)

base_recieved_outgoing.to_csv(output_dir / 'country_year_sanctions.csv', index=False)

In [71]:
crosswalk=base_recieved_outgoing[['gwcode', 'gw_name', 'iso3c']].groupby(['gwcode', 'gw_name', 'iso3c']).agg('first').reset_index()
crosswalk.to_csv('base_cross.csv')